# Data Quality: HVFHV - 8 Dimensiones - NYC TLC

Evaluacion de las 8 dimensiones de calidad de datos para el conjunto de datos HVFHV (High-Volume For-Hire Vehicle) del NYC Taxi and Limousine Commission.

| Dimension | Descripcion |
|---|---|
| Completitud | Presencia de valores en campos obligatorios |
| Exactitud | Valores concordantes con catalogos conocidos |
| Consistencia | Coherencia logica entre campos relacionados |
| Integridad | Integridad referencial con el catalogo de zonas TLC |
| Razonabilidad | Valores dentro de rangos esperados |
| Oportunidad | Datos dentro del rango temporal valido |
| Unicidad | Ausencia de registros duplicados |
| Validez | Formatos correctos en campos clave |

In [ ]:
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, '/home/jovyan/work')

from pyspark.sql import functions as F

# Tipo de vehiculo
VEHICLE = 'hvfhv'
RAW_PATH = f'/home/jovyan/work/data/raw/{VEHICLE}/'

# Diccionario para almacenar resultados de calidad
resultados_dq = {}

print(f'Tipo de vehiculo: {VEHICLE}')
print(f'Ruta de datos: {RAW_PATH}')

In [ ]:
from src.spark_utils import get_spark

spark = get_spark(app_name=f'DQ_{VEHICLE.upper()}')

# Lectura del conjunto de datos (lee todos los archivos fhvhv_*.parquet de la carpeta)
df = spark.read.parquet(RAW_PATH)

total_registros = df.count()
print(f'Total de registros cargados: {total_registros:,}')
print(f'Columnas disponibles: {df.columns}')
df.printSchema()

## Dimension 1: Completitud

**Pregunta:** Tenemos todos los datos necesarios?

Se verifica la presencia de valores no nulos en los campos obligatorios del esquema HVFHV.
El score es el promedio del porcentaje de completitud por campo obligatorio.

In [ ]:
# Campos obligatorios para HVFHV
campos_obligatorios = [
    'hvfhs_license_num',
    'dispatching_base_num',
    'pickup_datetime',
    'dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'base_passenger_fare',
    'driver_pay'
]

print('--- Dimension 1: Completitud ---')
print(f'Total de registros: {total_registros:,}')
print()

porcentajes_completitud = []
registros_fallidos_completitud = 0

for campo in campos_obligatorios:
    nulos = df.filter(F.col(campo).isNull()).count()
    pct_completo = ((total_registros - nulos) / total_registros) * 100
    porcentajes_completitud.append(pct_completo)
    registros_fallidos_completitud += nulos
    print(f'  Campo "{campo}": {nulos:,} nulos | {pct_completo:.2f}% completo')

# Score: promedio de completitud por campo
score_completitud = sum(porcentajes_completitud) / len(porcentajes_completitud)

resultados_dq['Completitud'] = {
    'score': round(score_completitud, 2),
    'descripcion': 'Porcentaje promedio de completitud en campos obligatorios',
    'registros_fallidos': registros_fallidos_completitud
}

print()
print(f'Score Completitud: {score_completitud:.2f}%')
print(f'Registros fallidos (suma de nulos por campo): {registros_fallidos_completitud:,}')

## Dimension 2: Exactitud

**Pregunta:** Los valores son correctos?

Se verifica que los valores coincidan con los catalogos conocidos del TLC:
- `hvfhs_license_num`: debe estar en ['HV0002', 'HV0003', 'HV0004', 'HV0005']
- `shared_request_flag`: debe ser 'Y' o 'N'
- `shared_match_flag`: debe ser 'Y' o 'N'
- `access_a_ride_flag`: debe ser 'Y' o 'N'
- `wav_request_flag`: debe ser 'Y' o 'N'
- `wav_match_flag`: debe ser 'Y' o 'N'

In [ ]:
print('--- Dimension 2: Exactitud ---')
print()

LICENCIAS_VALIDAS = ['HV0002', 'HV0003', 'HV0004', 'HV0005']
FLAGS_VALIDOS = ['Y', 'N']
COLUMNAS_FLAG = [
    'shared_request_flag',
    'shared_match_flag',
    'access_a_ride_flag',
    'wav_request_flag',
    'wav_match_flag'
]

# Regla 1: hvfhs_license_num en catalogo valido
falla_licencia = df.filter(
    F.col('hvfhs_license_num').isNotNull() &
    (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS))
).count()
print(f'  hvfhs_license_num fuera de catalogo {LICENCIAS_VALIDAS}: {falla_licencia:,} registros')

# Reglas de columnas de banderas (flags)
condicion_falla_flags = F.lit(False)
for col_flag in COLUMNAS_FLAG:
    falla_flag = df.filter(
        F.col(col_flag).isNotNull() &
        (~F.col(col_flag).isin(FLAGS_VALIDOS))
    ).count()
    print(f'  {col_flag} con valor invalido (no Y ni N): {falla_flag:,} registros')
    condicion_falla_flags = condicion_falla_flags | (
        F.col(col_flag).isNotNull() & (~F.col(col_flag).isin(FLAGS_VALIDOS))
    )

# Registros que fallan CUALQUIERA de las reglas de exactitud
condicion_falla_exactitud = (
    F.col('hvfhs_license_num').isNotNull() &
    (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS))
) | condicion_falla_flags

registros_fallidos_exactitud = df.filter(condicion_falla_exactitud).count()
score_exactitud = ((total_registros - registros_fallidos_exactitud) / total_registros) * 100

resultados_dq['Exactitud'] = {
    'score': round(score_exactitud, 2),
    'descripcion': 'Porcentaje de registros cuyos valores coinciden con catalogos TLC (licencia y banderas)',
    'registros_fallidos': registros_fallidos_exactitud
}

print()
print(f'Score Exactitud: {score_exactitud:.2f}%')
print(f'Registros fallidos: {registros_fallidos_exactitud:,}')

## Dimension 3: Consistencia

**Pregunta:** Los datos son coherentes entre si?

Se verifica la coherencia logica entre campos relacionados en HVFHV:
- `request_datetime` <= `pickup_datetime` <= `dropoff_datetime`
- `driver_pay` <= `base_passenger_fare` * 1.5 (participacion razonable del conductor)
- `trip_time` > 0 cuando `trip_miles` > 0

In [ ]:
print('--- Dimension 3: Consistencia ---')
print()

# Regla 1: request_datetime <= pickup_datetime
falla_request_pickup = df.filter(
    F.col('request_datetime').isNotNull() &
    F.col('pickup_datetime').isNotNull() &
    (F.col('request_datetime') > F.col('pickup_datetime'))
).count()
print(f'  request_datetime > pickup_datetime: {falla_request_pickup:,} registros')

# Regla 2: pickup_datetime <= dropoff_datetime
falla_orden_fechas = df.filter(
    F.col('pickup_datetime').isNotNull() &
    F.col('dropoff_datetime').isNotNull() &
    (F.col('pickup_datetime') >= F.col('dropoff_datetime'))
).count()
print(f'  pickup_datetime >= dropoff_datetime (orden incorrecto): {falla_orden_fechas:,} registros')

# Regla 3: driver_pay <= base_passenger_fare * 1.5
falla_driver_pay = df.filter(
    F.col('driver_pay').isNotNull() &
    F.col('base_passenger_fare').isNotNull() &
    (F.col('base_passenger_fare') > 0) &
    (F.col('driver_pay') > F.col('base_passenger_fare') * 1.5)
).count()
print(f'  driver_pay > base_passenger_fare * 1.5: {falla_driver_pay:,} registros')

# Regla 4: trip_time > 0 cuando trip_miles > 0
falla_trip_time_miles = df.filter(
    F.col('trip_miles').isNotNull() &
    F.col('trip_time').isNotNull() &
    (F.col('trip_miles') > 0) &
    (F.col('trip_time') <= 0)
).count()
print(f'  trip_time <= 0 cuando trip_miles > 0: {falla_trip_time_miles:,} registros')

# Registros que fallan CUALQUIERA de las reglas de consistencia
df_falla_consistencia = df.filter(
    (
        F.col('request_datetime').isNotNull() &
        F.col('pickup_datetime').isNotNull() &
        (F.col('request_datetime') > F.col('pickup_datetime'))
    ) |
    (
        F.col('pickup_datetime').isNotNull() &
        F.col('dropoff_datetime').isNotNull() &
        (F.col('pickup_datetime') >= F.col('dropoff_datetime'))
    ) |
    (
        F.col('driver_pay').isNotNull() &
        F.col('base_passenger_fare').isNotNull() &
        (F.col('base_passenger_fare') > 0) &
        (F.col('driver_pay') > F.col('base_passenger_fare') * 1.5)
    ) |
    (
        F.col('trip_miles').isNotNull() &
        F.col('trip_time').isNotNull() &
        (F.col('trip_miles') > 0) &
        (F.col('trip_time') <= 0)
    )
)
registros_fallidos_consistencia = df_falla_consistencia.count()
score_consistencia = ((total_registros - registros_fallidos_consistencia) / total_registros) * 100

resultados_dq['Consistencia'] = {
    'score': round(score_consistencia, 2),
    'descripcion': 'Porcentaje de registros con coherencia logica entre fechas, tarifas y kilometraje',
    'registros_fallidos': registros_fallidos_consistencia
}

print()
print(f'Score Consistencia: {score_consistencia:.2f}%')
print(f'Registros fallidos: {registros_fallidos_consistencia:,}')

## Dimension 4: Integridad

**Pregunta:** Se mantienen las relaciones entre tablas?

Se verifica la integridad referencial con el catalogo de zonas TLC y el catalogo de licencias:
- `PULocationID` debe estar en el rango 1-265 (zonas TLC validas) - UPPERCASE en HVFHV
- `DOLocationID` debe estar en el rango 1-265 (zonas TLC validas) - UPPERCASE en HVFHV
- `hvfhs_license_num` debe estar en el catalogo de licencias HVFHV validas

In [ ]:
print('--- Dimension 4: Integridad ---')
print()

# Rango valido de zonas TLC: 1 a 265
ZONA_MIN = 1
ZONA_MAX = 265
LICENCIAS_VALIDAS = ['HV0002', 'HV0003', 'HV0004', 'HV0005']

# Regla 1: PULocationID debe estar en rango valido (nota: UPPERCASE en HVFHV)
falla_pu = df.filter(
    F.col('PULocationID').isNotNull() &
    (
        (F.col('PULocationID') < ZONA_MIN) |
        (F.col('PULocationID') > ZONA_MAX)
    )
).count()
print(f'  PULocationID fuera de rango [1-265]: {falla_pu:,} registros')

# Regla 2: DOLocationID debe estar en rango valido (nota: UPPERCASE en HVFHV)
falla_do = df.filter(
    F.col('DOLocationID').isNotNull() &
    (
        (F.col('DOLocationID') < ZONA_MIN) |
        (F.col('DOLocationID') > ZONA_MAX)
    )
).count()
print(f'  DOLocationID fuera de rango [1-265]: {falla_do:,} registros')

# Regla 3: hvfhs_license_num en catalogo de licencias validas
falla_licencia_integridad = df.filter(
    F.col('hvfhs_license_num').isNotNull() &
    (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS))
).count()
print(f'  hvfhs_license_num fuera del catalogo HVFHV: {falla_licencia_integridad:,} registros')

# Registros que fallan CUALQUIERA de las reglas de integridad
df_falla_integridad = df.filter(
    (
        F.col('PULocationID').isNotNull() &
        ((F.col('PULocationID') < ZONA_MIN) | (F.col('PULocationID') > ZONA_MAX))
    ) |
    (
        F.col('DOLocationID').isNotNull() &
        ((F.col('DOLocationID') < ZONA_MIN) | (F.col('DOLocationID') > ZONA_MAX))
    ) |
    (
        F.col('hvfhs_license_num').isNotNull() &
        (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS))
    )
)
registros_fallidos_integridad = df_falla_integridad.count()
score_integridad = ((total_registros - registros_fallidos_integridad) / total_registros) * 100

resultados_dq['Integridad'] = {
    'score': round(score_integridad, 2),
    'descripcion': 'Porcentaje de registros con IDs de zona en rango TLC valido y licencia en catalogo',
    'registros_fallidos': registros_fallidos_integridad
}

print()
print(f'Score Integridad: {score_integridad:.2f}%')
print(f'Registros fallidos: {registros_fallidos_integridad:,}')

## Dimension 5: Razonabilidad

**Pregunta:** Los valores estan dentro de rangos esperados?

Se verifica que los valores numericos clave esten dentro de rangos razonables para HVFHV:
- `trip_miles`: entre 0 y 300 millas (Uber/Lyft pueden tener viajes largos)
- `base_passenger_fare`: entre 0 y 1000 dolares
- `driver_pay`: entre 0 y 800 dolares
- `trip_time`: entre 60 y 43,200 segundos (1 minuto a 12 horas)
- `tips`: mayor o igual a 0

In [ ]:
print('--- Dimension 5: Razonabilidad ---')
print()

# Regla 1: trip_miles entre 0 y 300
falla_millas = df.filter(
    F.col('trip_miles').isNotNull() &
    (
        (F.col('trip_miles') < 0) |
        (F.col('trip_miles') > 300)
    )
).count()
print(f'  trip_miles fuera de rango [0-300]: {falla_millas:,} registros')

# Regla 2: base_passenger_fare entre 0 y 1000
falla_tarifa = df.filter(
    F.col('base_passenger_fare').isNotNull() &
    (
        (F.col('base_passenger_fare') < 0) |
        (F.col('base_passenger_fare') > 1000)
    )
).count()
print(f'  base_passenger_fare fuera de rango [0-1000]: {falla_tarifa:,} registros')

# Regla 3: driver_pay entre 0 y 800
falla_driver_pay = df.filter(
    F.col('driver_pay').isNotNull() &
    (
        (F.col('driver_pay') < 0) |
        (F.col('driver_pay') > 800)
    )
).count()
print(f'  driver_pay fuera de rango [0-800]: {falla_driver_pay:,} registros')

# Regla 4: trip_time entre 60 y 43200 segundos
falla_trip_time = df.filter(
    F.col('trip_time').isNotNull() &
    (
        (F.col('trip_time') < 60) |
        (F.col('trip_time') > 43200)
    )
).count()
print(f'  trip_time fuera de rango [60-43200 segundos]: {falla_trip_time:,} registros')

# Regla 5: tips >= 0
falla_tips = df.filter(
    F.col('tips').isNotNull() &
    (F.col('tips') < 0)
).count()
print(f'  tips negativos: {falla_tips:,} registros')

# Registros que fallan CUALQUIERA de las reglas de razonabilidad
df_falla_razon = df.filter(
    (
        F.col('trip_miles').isNotNull() &
        ((F.col('trip_miles') < 0) | (F.col('trip_miles') > 300))
    ) |
    (
        F.col('base_passenger_fare').isNotNull() &
        ((F.col('base_passenger_fare') < 0) | (F.col('base_passenger_fare') > 1000))
    ) |
    (
        F.col('driver_pay').isNotNull() &
        ((F.col('driver_pay') < 0) | (F.col('driver_pay') > 800))
    ) |
    (
        F.col('trip_time').isNotNull() &
        ((F.col('trip_time') < 60) | (F.col('trip_time') > 43200))
    ) |
    (
        F.col('tips').isNotNull() &
        (F.col('tips') < 0)
    )
)
registros_fallidos_razon = df_falla_razon.count()
score_razonabilidad = ((total_registros - registros_fallidos_razon) / total_registros) * 100

resultados_dq['Razonabilidad'] = {
    'score': round(score_razonabilidad, 2),
    'descripcion': 'Porcentaje de registros con valores numericos dentro de rangos razonables',
    'registros_fallidos': registros_fallidos_razon
}

print()
print(f'Score Razonabilidad: {score_razonabilidad:.2f}%')
print(f'Registros fallidos: {registros_fallidos_razon:,}')

## Dimension 6: Oportunidad

**Pregunta:** Los datos estan actualizados?

Se verifica que los registros correspondan al rango temporal esperado para el conjunto de datos HVFHV:
- Rango valido: 2019-2025 (Ley HVFHV entro en vigor en febrero de 2019)
- Se contabilizan registros por anio para identificar la distribucion temporal

In [ ]:
print('--- Dimension 6: Oportunidad ---')
print()

ANIO_MIN = 2019
ANIO_MAX = 2025

# Extraer anio del pickup_datetime
df_con_anio = df.withColumn('anio_recogida', F.year('pickup_datetime'))

# Distribucion por anio
print('  Distribucion de registros por anio de recogida:')
dist_anio = df_con_anio.groupBy('anio_recogida').count().orderBy('anio_recogida')
dist_anio.show(truncate=False)

# Registros fuera del rango valido
registros_fallidos_oportunidad = df_con_anio.filter(
    F.col('anio_recogida').isNull() |
    (F.col('anio_recogida') < ANIO_MIN) |
    (F.col('anio_recogida') > ANIO_MAX)
).count()

score_oportunidad = ((total_registros - registros_fallidos_oportunidad) / total_registros) * 100

resultados_dq['Oportunidad'] = {
    'score': round(score_oportunidad, 2),
    'descripcion': f'Porcentaje de registros con anio de recogida en el rango [{ANIO_MIN}-{ANIO_MAX}]',
    'registros_fallidos': registros_fallidos_oportunidad
}

print(f'Registros fuera del rango [{ANIO_MIN}-{ANIO_MAX}]: {registros_fallidos_oportunidad:,}')
print()
print(f'Score Oportunidad: {score_oportunidad:.2f}%')

## Dimension 7: Unicidad

**Pregunta:** Existen registros duplicados?

Se detectan registros duplicados agrupando por la combinacion de campos que identifican unicamente un viaje HVFHV:
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `hvfhs_license_num`
- `base_passenger_fare`

In [ ]:
print('--- Dimension 7: Unicidad ---')
print()

# Campos que definen la unicidad de un viaje HVFHV
campos_unicidad = [
    'pickup_datetime',
    'dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'hvfhs_license_num',
    'base_passenger_fare'
]

# Contar ocurrencias por combinacion de campos
df_agrupado = df.groupBy(campos_unicidad).count()

# Registros duplicados: grupos con mas de 1 ocurrencia
df_duplicados = df_agrupado.filter(F.col('count') > 1)
grupos_duplicados = df_duplicados.count()

# Total de registros excedentes que son duplicados
excedentes_resultado = df_duplicados.agg(
    F.sum(F.col('count') - 1).alias('excedentes')
).collect()[0]['excedentes']

registros_duplicados_total = int(excedentes_resultado) if excedentes_resultado is not None else 0

registros_unicos = total_registros - registros_duplicados_total
score_unicidad = (registros_unicos / total_registros) * 100

print(f'  Grupos de registros duplicados encontrados: {grupos_duplicados:,}')
print(f'  Total de registros excedentes (duplicados): {registros_duplicados_total:,}')
print(f'  Registros unicos: {registros_unicos:,}')

resultados_dq['Unicidad'] = {
    'score': round(score_unicidad, 2),
    'descripcion': 'Porcentaje de registros unicos (sin duplicados) por combinacion de campos clave',
    'registros_fallidos': registros_duplicados_total
}

print()
print(f'Score Unicidad: {score_unicidad:.2f}%')

## Dimension 8: Validez

**Pregunta:** Los formatos son correctos?

Se verifica que los campos clave tengan formatos validos en HVFHV:
- `dispatching_base_num`: debe comenzar con 'B' (validacion de formato)
- `hvfhs_license_num`: debe coincidir con el patron HV seguido de 4 digitos (HV[0-9]{4})
- Columnas de banderas (`shared_request_flag`, etc.): deben tener exactamente 1 caracter
- `trip_time`: debe ser un entero (sin decimales que indiquen problemas de tipo)

In [ ]:
print('--- Dimension 8: Validez ---')
print()

COLUMNAS_FLAG = [
    'shared_request_flag',
    'shared_match_flag',
    'access_a_ride_flag',
    'wav_request_flag',
    'wav_match_flag'
]

# Regla 1: dispatching_base_num debe comenzar con 'B'
falla_formato_base = df.filter(
    F.col('dispatching_base_num').isNotNull() &
    (~F.col('dispatching_base_num').startswith('B'))
).count()
print(f'  dispatching_base_num no comienza con "B": {falla_formato_base:,} registros')

# Regla 2: hvfhs_license_num debe coincidir con patron HV[0-9]{4}
falla_patron_licencia = df.filter(
    F.col('hvfhs_license_num').isNotNull() &
    (~F.col('hvfhs_license_num').rlike('^HV[0-9]{4}$'))
).count()
print(f'  hvfhs_license_num no coincide con patron HV[0-9]{{4}}: {falla_patron_licencia:,} registros')

# Regla 3: columnas de banderas deben tener exactamente 1 caracter
condicion_falla_long_flags = F.lit(False)
for col_flag in COLUMNAS_FLAG:
    falla_len_flag = df.filter(
        F.col(col_flag).isNotNull() &
        (F.length(F.col(col_flag)) != 1)
    ).count()
    print(f'  {col_flag} con longitud != 1: {falla_len_flag:,} registros')
    condicion_falla_long_flags = condicion_falla_long_flags | (
        F.col(col_flag).isNotNull() & (F.length(F.col(col_flag)) != 1)
    )

# Regla 4: trip_time debe ser un valor entero (sin parte decimal)
falla_trip_time_decimal = df.filter(
    F.col('trip_time').isNotNull() &
    (F.col('trip_time') != F.floor(F.col('trip_time')))
).count()
print(f'  trip_time con parte decimal (no entero): {falla_trip_time_decimal:,} registros')

# Registros que fallan CUALQUIERA de las reglas de validez
condicion_falla_validez = (
    (
        F.col('dispatching_base_num').isNotNull() &
        (~F.col('dispatching_base_num').startswith('B'))
    ) |
    (
        F.col('hvfhs_license_num').isNotNull() &
        (~F.col('hvfhs_license_num').rlike('^HV[0-9]{4}$'))
    ) |
    condicion_falla_long_flags |
    (
        F.col('trip_time').isNotNull() &
        (F.col('trip_time') != F.floor(F.col('trip_time')))
    )
)
registros_fallidos_validez = df.filter(condicion_falla_validez).count()
score_validez = ((total_registros - registros_fallidos_validez) / total_registros) * 100

resultados_dq['Validez'] = {
    'score': round(score_validez, 2),
    'descripcion': 'Porcentaje de registros con formatos correctos en campos clave (base, licencia, banderas)',
    'registros_fallidos': registros_fallidos_validez
}

print()
print(f'Score Validez: {score_validez:.2f}%')
print(f'Registros fallidos: {registros_fallidos_validez:,}')

## Resumen Final: Puntuacion Global de Calidad de Datos

Se presenta el resumen de las 8 dimensiones evaluadas con su puntuacion individual y la puntuacion global promedio.

**Escala de colores:**
- Verde: score >= 95% (calidad alta)
- Naranja: score >= 80% (calidad media)
- Rojo: score < 80% (calidad baja)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

print('=' * 60)
print('RESUMEN FINAL: CALIDAD DE DATOS HVFHV')
print('=' * 60)
print()

# Construir lista de resultados
filas_resumen = []
for dimension, valores in resultados_dq.items():
    filas_resumen.append((dimension, float(valores['score']), int(valores['registros_fallidos'])))
    print(f'  {dimension:<18} | Score: {valores["score"]:6.2f}% | Fallidos: {valores["registros_fallidos"]:>12,}')

print()

# Score global (promedio de las 8 dimensiones)
scores = [v['score'] for v in resultados_dq.values()]
score_global = sum(scores) / len(scores)
print(f'  Score Global (promedio): {score_global:.2f}%')
print()

# Crear DataFrame de resumen con PySpark
schema_resumen = StructType([
    StructField('Dimension', StringType(), True),
    StructField('Score', FloatType(), True),
    StructField('Registros_Fallidos', IntegerType(), True)
])
df_resumen = spark.createDataFrame(filas_resumen, schema=schema_resumen)
print('Tabla de resultados:')
df_resumen.show(truncate=False)

# --- Grafico de barras horizontales ---
dimensiones = [f[0] for f in filas_resumen]
scores_lista = [f[1] for f in filas_resumen]

colores = []
for s in scores_lista:
    if s >= 95:
        colores.append('#2ecc71')   # Verde
    elif s >= 80:
        colores.append('#f39c12')   # Naranja
    else:
        colores.append('#e74c3c')   # Rojo

fig, ax = plt.subplots(figsize=(11, 6))
y_pos = range(len(dimensiones))
barras = ax.barh(y_pos, scores_lista, color=colores, edgecolor='white', height=0.6)

# Etiquetas de valor en las barras
for barra, score in zip(barras, scores_lista):
    ax.text(
        barra.get_width() + 0.3,
        barra.get_y() + barra.get_height() / 2,
        f'{score:.1f}%',
        va='center',
        ha='left',
        fontsize=10,
        fontweight='bold'
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(dimensiones, fontsize=11)
ax.set_xlabel('Score de Calidad (%)', fontsize=11)
ax.set_title(
    f'Calidad de Datos HVFHV - 8 Dimensiones\nScore Global: {score_global:.2f}%',
    fontsize=13,
    fontweight='bold'
)
ax.set_xlim(0, 108)
ax.axvline(x=95, color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1)
ax.axvline(x=80, color='#f39c12', linestyle='--', alpha=0.6, linewidth=1)

# Leyenda
leyenda = [
    mpatches.Patch(color='#2ecc71', label='Alta (>= 95%)'),
    mpatches.Patch(color='#f39c12', label='Media (>= 80%)'),
    mpatches.Patch(color='#e74c3c', label='Baja (< 80%)')
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/home/jovyan/work/data/outputs/dq_hvfhv_8_dimensiones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico guardado en: /home/jovyan/work/data/outputs/dq_hvfhv_8_dimensiones.png')

spark.stop()
print()
print('Sesion de Spark finalizada.')